# 02. Agent Architecture의 변화

Multi-Agent 구조의 이름은 매우 다양합니다. 이 노트북에서는 구조의 이름을 단순히 외우는 대신 한 가지 핵심 질문으로 각 구조를 살펴봅니다. **이 구조는 직전 구조의 어떤 한계 때문에 등장했고, 그 대가로 무엇을 새로 치르는가?** 어떤 구조도 공짜는 없습니다. 따라서 아키텍처 변화의 방향은 무조건 "더 나은 쪽"이 아니라 "다른 한계를 선택한 쪽"입니다.

이번 실습에서는 동일한 고객 요청 T-1001("결제가 두 번 됐어요. 취소하고 보상안도 함께 확인해 주세요.")을 Single, Supervisor, Handoff 구조로 실제 실행해 보고, 나머지 구조들은 일반 함수로 뼈대만 구성하여 제어 흐름을 확인합니다. 결제 내역, 가용률, 규정은 모두 수업용 가상 데이터이며 저장소 외부로 어떠한 데이터도 전송되지 않습니다.

## 학습 목표

- 아키텍처의 변화를 직전 구조의 한계 관점에서 설명합니다(Single → Skill → Supervisor → Orchestrator-Worker → Handoff → Hierarchical).
- Single 구조의 Context가 모델 호출마다 어떻게 커지는지 실측하고, 어떤 요소가 커지는지 분해하여 확인합니다.
- Deep Agents의 `subagents=` 설정으로 Supervisor 구조를 실행하고, "모델이 결정한 병렬 실행"을 trace로 확인합니다.
- Network와 Handoff의 차이를 구분하고, `Command(goto=)`를 사용하여 담당자(owner) 이전과 왕복 상한을 구현합니다.
- 동일한 요청에 대한 실제 실행 trace에서 제어권이 달라지는 지점을 찾아 비교합니다.
- 프레임워크가 자동으로 추가한 Worker와 직접 설계한 Worker의 차이를 코드 객체에서 추출한 표로 구분합니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 1 | 변화의 지도 | 6 |
| 2 | Single의 한계 실측 | 9 |
| 3 | Supervisor: 결과를 돌려받는다 | 17 |
| 4 | Handoff: 대화를 넘긴다 | 12 |
| 5 | Supervisor, Orchestrator-Worker, Hierarchical, Plan-and-Execute 읽기 | 7 |
| 6 | 실제 Trace 비교 | 4 |
| 7 | 직접 해보기와 정리 | 5 |

## 호출 규모

기본 실행에 포함된 유료 셀은 5개이며, 모델 호출은 약 25~60회로 추정됩니다. EXA 웹 검색 호출은 없습니다. `직접 해보기`의 유료 셀 2개는 기본적으로 꺼져 있으며(`RUN_EXERCISE = False`), 활성화하면 모델 호출이 약 14~26회 추가됩니다. 각 유료 셀은 호출 직전에 예상 규모를 한 줄로 고지하므로, 본인의 크레딧이 사용되는 실행임을 확인하고 진행해 주세요. 나머지 셀은 일반 함수로 동작하며 대부분 출력에 "모델 호출 0회"를 표시합니다. 환경 준비 셀이나 이전 실행의 호출표를 읽기만 하는 셀은 별도 문구가 없더라도 모델을 호출하지 않습니다.

아래 코드는 실습에 필요한 패키지와 모듈을 불러오고 API 키 및 기본 설정을 확인합니다.

In [ ]:
# 표준 라이브러리 불러오기
import inspect, json, re, time
from operator import add
from typing import Annotated, Literal, TypedDict
from uuid import uuid4

# Deep Agents 및 LangChain/LangGraph 핵심 모듈 불러오기
from deepagents import create_deep_agent
from deepagents.middleware.subagents import GENERAL_PURPOSE_SUBAGENT
from langchain.agents.structured_output import ProviderStrategy
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command

# 실습용 패키지(mas_lab, rag_lab) 모듈 불러오기
from mas_lab.contracts import FinalAnswer, WorkerTask, validate_final, validate_result
from mas_lab.desk import Desk, load_outage
from mas_lab.harness import framework_default_subagent, task_targets
from mas_lab.runner import make_desk
from mas_lab.structures import MAX_HANDOFFS, OUTCOME_GUIDE, build_handoff, build_single, build_supervisor, gate_answer, scope_line
from mas_lab.trace import ModelMeter, overlap_seconds, timeline
from mas_lab.view import bars, flows_of, message_table, show_graph, show_table, write_design_log
from mas_lab.workers import ROLE_TOOLS, allowlist_middleware, run_worker, subagent_spec
from rag_lab.corpus import load_cases
from rag_lab.settings import announce, key_status, make_model, require_key

# API 키 상태 확인 및 모델 초기화
print(key_status())
settings = require_key()  # 키가 설정되지 않은 경우 한국어 안내와 함께 여기서 실행이 멈춥니다.
model = make_model()      # 모델 객체를 생성하는 단계이며, 아직 실제 API 호출은 발생하지 않습니다.

# 실습에 사용할 고객 문의 사례 데이터 로드
CASES = {case["id"]: case for case in load_cases()}
double_pay, followup = CASES["complex-double-pay"], CASES["followup-handoff"]
print("모델:", settings.model, "| 사례:", double_pay["id"], "|", double_pay["question"], "| Desk:", Desk.__name__, "| 측정:", ModelMeter.__name__)

아래 셀은 이 노트북에서 공통으로 사용할 측정 및 실행 도우미를 정의합니다. 기본 `ModelMeter`는 모델 호출별 토큰 수와 소요 시간을 기록합니다. 이번 실습에서는 여기에 세 가지 정보를 더 기록해야 합니다. 바로 **어느 Agent가 호출했는지**, **마지막 호출에 어떤 입력이 들어갔는지**, 그리고 **`task` 도구가 언제 시작해 언제 끝났는지**입니다.

특히 세 번째 항목이 중요합니다. `run_worker`와 Handoff의 담당자(owner) 노드는 `desk.trace.span`을 기록하고, `build_supervisor`는 `delegation_spans`를 통해 `task` 호출마다 실행 구간을 기록합니다. 그럼에도 여기서 콜백(callback)으로 `task` 구간을 한 번 더 독립적으로 수집합니다. 측정 장치가 없으면 동시 실행 겹침(overlap)은 언제나 0으로 나타나기 때문에, 실제로 "겹치지 않은 것"과 "측정하지 않은 것"을 정확히 구분하려면 코드 내부 기록과 별개로 동작하는 측정 기록이 필요합니다.

아래 코드는 호출 주체와 도구 실행 구간을 함께 수집하는 `Recorder` 클래스, 실행 도우미 함수, 그리고 결과 비교표 생성 함수를 차례대로 정의합니다.

In [ ]:
class Recorder(ModelMeter):
    '''ModelMeter에 호출 주체, 마지막 입력, task 실행 구간, 도구 결과 크기를 추가로 기록합니다. 모델 응답 자체는 변경하지 않습니다.'''
    def __init__(self, desk: Desk):
        super().__init__()
        # 측정 결과를 담을 상태 변수 초기화
        self.desk, self.spans, self.tool_results, self.last_messages, self.last_params, self._tools = desk, [], [], [], {}, {}

    def on_chat_model_start(self, serialized, messages, *, run_id, **kwargs):
        # 모델 호출 시작 시 호출 주체(lc_agent_name)와 마지막 입력 메시지·파라미터를 기록합니다.
        super().on_chat_model_start(serialized, messages, run_id=run_id, **kwargs)
        with self.lock:
            self._open[run_id]["agent"] = (kwargs.get("metadata") or {}).get("lc_agent_name") or "triage"
            self.last_messages, self.last_params = messages[0], kwargs.get("invocation_params", {})

    def on_tool_start(self, serialized, input_str, *, run_id, inputs=None, **kwargs):
        # 도구 호출 시작 시각과 대상 Worker 정보를 기록합니다.
        with self.lock:
            self._tools[run_id] = {"tool": kwargs.get("name") or (serialized or {}).get("name"), "worker": (inputs or {}).get("subagent_type"),
                                   "agent": (kwargs.get("metadata") or {}).get("lc_agent_name"),
                                   "start": round(time.perf_counter() - self.desk.trace.started, 3)}

    def on_tool_end(self, output, *, run_id, **kwargs):
        # 도구 호출 종료 시 task 도구이면 실행 구간(span)을 추가하고, 일반 도구이면 결과 글자 수를 기록합니다.
        with self.lock:
            tool = self._tools.pop(run_id, None)
            if tool and tool["tool"] == "task":
                self.spans.append({"actor": tool["worker"], "kind": "worker", "start": tool["start"],
                                   "end": round(time.perf_counter() - self.desk.trace.started, 3)})
            elif tool:
                self.tool_results.append({**tool, "chars": len(str(getattr(output, "content", output)))})

    def on_tool_error(self, error, *, run_id, **kwargs):
        # 도구 실행 중 오류가 발생해도 종료 처리를 수행하여 구간 기록이 누락되지 않게 합니다.
        self.on_tool_end(type(error).__name__, run_id=run_id)

print("측정 도우미:", Recorder.__name__, "| 추가 기록 항목:", ["agent", "spans", "tool_results", "last_messages", "last_params"], "| 모델 호출 0회")

In [ ]:
def execute(app, payload, recorder, thread_id=None):
    '''어느 아키텍처든 동일한 인터페이스로 1회 실행합니다. 실행 중 예외가 발생하더라도 중단하지 않고 오류 내용을 결과 행에 기록합니다.'''
    config = {"callbacks": [recorder], "recursion_limit": 60, "configurable": {"thread_id": thread_id or uuid4().hex}}
    started, state, error = time.perf_counter(), {}, ""
    try:
        state = app.invoke(payload, config=config)
    except Exception as failure:  # 노트북 실행이 멈추지 않으면서 실패 원인이 비교표에 남아야 합니다. API 429 오류나 예산(budget) 초과도 여기서 포착합니다.
        error = f"{type(failure).__name__}: {str(failure)[:200]}"
    # 구조화된 최종 응답(structured_response 또는 final) 추출 및 검증
    raw = state.get("structured_response") or state.get("final")
    final = FinalAnswer.model_validate(raw) if isinstance(raw, dict) else raw
    return {"state": state, "final": final, "error": error, "seconds": round(time.perf_counter() - started, 1), "rec": recorder}

def ask(question):
    # 사용자 질문 문자열을 LangGraph 메시지 입력 형식으로 변환합니다.
    return {"messages": [{"role": "user", "content": question}]}

def call_rows(recorder):
    # 모델 호출 기록을 표 형태로 출력하기 위한 행 리스트로 변환합니다.
    return [{"순번": i + 1, "주체": c.get("agent"), "입력 토큰": c["input_tokens"], "캐시 읽기": c["cache_read_tokens"], "출력 토큰": c["output_tokens"],
             "메시지 수": c["messages"], "시간(초)": round(c["end"] - c["start"], 2) if c["end"] else None} for i, c in enumerate(recorder.calls)]

def task_rows(messages):
    # 부모 에이전트의 메시지 목록에서 task 도구(Worker 위임) 호출 내역만 추출합니다.
    rows = []
    for index, message in enumerate(messages):
        calls = [c for c in getattr(message, "tool_calls", []) if c["name"] == "task"]
        if calls:
            rows.append({"메시지 순번": index, "task 수": len(calls), "위임 대상": ", ".join(c["args"].get("subagent_type", "?") for c in calls)})
    return rows

print("실행 도우미:", [f.__name__ for f in (execute, ask, call_rows, task_rows)], "| 모델 호출 0회")

In [ ]:
def credit_basis(desk, events):
    '''서비스 크레딧을 어떤 기준으로 계산했는지 확인합니다. calculate_credit 도구의 인자를 실제 결제 내역의 기본 이용료분과 대조합니다.'''
    base_fee = next(p["base_fee_part"] for p in desk.payments() if p["payment_id"] == "P-9002")
    calls = [e for e in events if e["kind"] == "tool" and e["actor"] == "calculate_credit"]
    if not calls:
        return "계산 도구 호출 없음"
    def read(e):
        verdict = "기본 이용료분과 일치" if e["base_fee"] == base_fee else f"기본 이용료분 {base_fee:,}과 다름"
        return f"{e['base_fee']:,} x {e['credit_percent']}% ({verdict})"
    return " / ".join(map(read, calls))

def amounts_in(final):
    # 최종 응답 본문에서 원화 금액 표기(예: 1,200,000원)를 모두 추출합니다.
    return re.findall(r"\d{1,3}(?:,\d{3})+원", final.text) if final else []

def row_of(name, run, desk, events, owners, speaker, scope_text, passed_chars):
    '''세 가지 아키텍처의 실행 결과를 동일한 컬럼 구조로 정리합니다. 실패한 실행 행도 누락 없이 포함합니다.'''
    totals, final = run["rec"].totals(), run["final"]
    gated = gate_answer(final, desk, request_id=name) if final else None  # 실제 쓰기 경로는 어느 구조에서나 이 게이트 하나를 통과합니다.
    return {"구조": name, "control owner": " → ".join(owners), "모델 호출": totals["model_calls"], "입력 토큰": totals["input_tokens"],
            "캐시 읽기": totals["cache_read_tokens"], "usage 누락": totals["usage_missing"], "시간(초)": run["seconds"],
            "Context 전달 범위": scope_text, "전달 글자 수": passed_chars, "최종 응답 주체": speaker, "outcome": gated.outcome if gated else None,
            "크레딧 계산 기준": credit_basis(desk, events), "답의 금액": sorted(set(amounts_in(final))),
            "검증 문제": validate_final(final, desk.known_refs(), desk.source_text()) if final else ["최종 응답 없음"], "실행 오류": run["error"]}

print("읽기 도우미:", [f.__name__ for f in (credit_basis, amounts_in, row_of)], "| 모델 호출 0회")

## 1. 변화의 지도

**문제.** 아키텍처의 이름부터 먼저 배우면 "우리 업무에 어떤 구조가 적합한가?"라는 질문이 쉽게 "요즘 어떤 구조가 최신인가?"라는 질문으로 바뀝니다. 따라서 아래 표에서는 각 구조를 단순한 이름이 아니라 **직전 구조의 한계와 새로 치르는 비용**을 기준으로 정리합니다.

| 구조 | 직전 구조의 어떤 한계에서 나왔는가 | 새로 치르는 비용 |
|---|---|---|
| Single | 고정 경로는 예외 상황을 미리 모두 정의해야 하므로, 관찰 후 다음 행동을 모델에게 맡깁니다 | 호출 횟수와 순서를 미리 알 수 없고, 모든 정보가 하나의 Context에 쌓입니다 |
| Skill | Single의 시스템 지침에 업무 절차를 모두 넣으면 지침이 길어지고, 해당 절차가 필요 없는 요청에도 매번 토큰 비용이 듭니다 | 절차를 읽을지 여부를 모델이 결정하므로, 읽지 않았거나 일부만 읽었는지 별도로 확인해야 합니다 |
| Supervisor | Skill은 절차만 분리할 뿐 조사 결과는 여전히 하나의 Context에 쌓이므로, 부모 모델이 조사를 Worker에게 위임하고 결과만 돌려받습니다 | Worker마다 모델 호출이 늘어나고, Worker가 반환하지 않은 정보는 부모가 알 수 없으며, 위임 순서와 병렬 여부도 모델의 판단에 좌우됩니다 |
| Orchestrator-Worker | Supervisor의 위임 순서와 병렬 여부는 모델의 선택과 프롬프트 한 줄에 흔들리므로, 작업을 코드가 처음에 나누어 한꺼번에 보내고 결과를 모읍니다 | 처음에 나눈 방식이 틀리면 전체가 틀려지며, 상태와 경로를 직접 설계하고 결과 병합 및 부분 실패를 처리해야 합니다 |
| Handoff | 앞선 구조들은 모두 한 주체가 결과를 모아 응답하지만, 대화 자체가 다른 전문 영역으로 넘어가면 해당 담당자(owner)가 이어서 응답하게 합니다 | 넘겨준 메모에 없는 사실은 사라지며, 잘못 유지된 담당자나 무한 왕복이 새로운 실패 원인이 됩니다 |
| Hierarchical | Worker가 많아지면 단일 Supervisor의 Context와 판단력이 한계에 닿으므로, 팀 단위로 묶고 요약만 상위 계층으로 올립니다 | 계층을 거칠 때마다 정보 전달 손실과 응답 지연이 더해집니다 |
| Plan-and-Execute | 한 걸음씩 다음 행동을 고르는 구조는 전체 일정이 보이지 않으므로, 먼저 전체 계획을 세우고 차례대로 실행합니다 | 초기 계획이 틀렸을 때 중간에 수정할 기회를 만들지 않으면 끝까지 비용과 시간을 낭비합니다 |

**용어 정리.** 부모 에이전트가 Worker를 도구처럼 호출하고 결과를 돌려받아 최종 응답을 작성하는 구조를 넓게는 Manager 패턴이라고 부릅니다. 이 과정에서는 라이브러리의 관례에 따라 이 구조를 **Supervisor**라고 부릅니다. `langgraph-supervisor` 패키지의 `create_supervisor`가 이 이름을 사용하며, Deep Agents의 `subagents=` 옵션과 이 과정의 `build_supervisor` 함수가 모두 동일한 형태입니다. 다음 Worker를 선택하는 역할을 모델 대신 그래프의 라우팅 함수가 맡더라도 제어 흐름의 모양은 동일합니다. 이 구조의 기본 골격은 5절에서 일반 함수 뼈대로 다시 확인합니다.

위 표의 흐름은 구식 기술을 버리는 세대교체가 아니라 **선택지가 넓어진 것**입니다. 뒤에 나오는 구조가 앞선 구조를 무조건 대체하지 않습니다. 앞선 구조의 한계를 아직 겪지 않았다면 단순한 구조가 훨씬 저렴하고 디버깅하기도 쉽습니다. 이번 노트북에서는 위 표의 "새로 치르는 비용" 항목을 직접 실행하고 수치로 확인합니다.

아래 코드에서는 일곱 가지 아키텍처의 제어권과 응답 주체를 요약한 표를 출력합니다.

In [ ]:
# 일곱 가지 아키텍처의 핵심 제어 흐름과 이 노트북에서의 실행 방식을 표로 정리합니다.
ARCHITECTURES = [
    {"구조": "Single", "다음 행동 결정": "단일 모델", "결과가 돌아가는 곳": "자신의 Context", "최종 응답자": "해당 Agent", "이 노트북의 실행 여부": "실제 모델 실행"},
    {"구조": "Skill", "다음 행동 결정": "단일 모델 (절차는 파일에서 읽음)", "결과가 돌아가는 곳": "자신의 Context", "최종 응답자": "해당 Agent", "이 노트북의 실행 여부": "표로만 비교"},
    {"구조": "Supervisor", "다음 행동 결정": "부모 모델이 task로 위임 (5절 뼈대는 route 함수)", "결과가 돌아가는 곳": "부모 Agent", "최종 응답자": "부모 Agent", "이 노트북의 실행 여부": "실제 모델 실행 (5절은 일반 함수 뼈대)"},
    {"구조": "Orchestrator-Worker", "다음 행동 결정": "분해 단계에서 한꺼번에 결정", "결과가 돌아가는 곳": "결과 병합 노드", "최종 응답자": "결과 병합 노드", "이 노트북의 실행 여부": "표로만 비교 (03 노트북에서 실행)"},
    {"구조": "Handoff", "다음 행동 결정": "현재 담당자(owner)", "결과가 돌아가는 곳": "되돌아가지 않고 대화 제어권이 넘어감", "최종 응답자": "마지막 담당자(owner)", "이 노트북의 실행 여부": "실제 모델 실행"},
    {"구조": "Hierarchical", "다음 행동 결정": "각 계층에서 개별 결정", "결과가 돌아가는 곳": "바로 위 상위 계층", "최종 응답자": "최상위 계층", "이 노트북의 실행 여부": "일반 함수 뼈대 실행"},
    {"구조": "Plan-and-Execute", "다음 행동 결정": "계획 수립 단계에서 미리 결정", "결과가 돌아가는 곳": "실행기 (재계획 시 계획자)", "최종 응답자": "실행기", "이 노트북의 실행 여부": "일반 함수 뼈대 실행"},
]
show_table(ARCHITECTURES)
print("모델 호출 0회. 표의 항목을 수정하면 마지막 설계 기록 셀의 선택지도 함께 바뀝니다.")

## 2. Single의 한계 실측

**문제.** "Single 구조는 Context가 계속 커져서 한계가 있다"라고 말하기는 쉽습니다. 하지만 실제로 호출마다 얼마나 커지는지, 그리고 무엇 때문에 커지는지 측정하지 않으면 Multi-Agent 구조를 도입할 객관적인 근거가 되지 못합니다. 한계를 실측하지 않고 구조만 복잡하게 바꾸면 오히려 비용만 증가합니다.

**먼저 예측하세요.** T-1001 요청을 Single 구조로 실행하면 마지막 모델 호출의 입력 토큰은 첫 번째 호출의 몇 배가 될까요? 또한 시스템 지침, 도구 스키마, 대화 이력, 도구 결과 중 어느 항목이 가장 큰 비중을 차지할까요? 예상되는 배수와 항목 이름을 먼저 적어 본 뒤 아래 코드 셀을 차례대로 실행하여 확인합니다.

In [ ]:
# Single 에이전트로 T-1001 복합 문의를 실행하고 소요 시간과 호출된 도구 목록을 확인합니다.
announce("Single 에이전트가 T-1001 요청 처리", model_calls=12)
single_desk = make_desk(double_pay, label="single")  # 시간 예산(budget)은 Desk를 생성할 때 측정이 시작되므로 실행 직전에 생성합니다.
single_app = build_single(single_desk, model)
single = execute(single_app, ask(double_pay["question"]), Recorder(single_desk))
print("실행 오류:", single["error"] or "없음", "| 시간(초):", single["seconds"], "| outcome:", single["final"].outcome if single["final"] else None)
print("호출된 도구:", [e["name"] for e in single_desk.budget.report()["events"] if e["event"] == "tool"])

In [ ]:
# Single 실행에서 발생한 모델 호출별 토큰 추이와 막대 그래프를 출력합니다.
rows = call_rows(single["rec"])
show_table(rows)
print(bars({f"호출 {r['순번']}": r["입력 토큰"] or 0 for r in rows}))
print("usage 누락:", single["rec"].totals()["usage_missing"], "건 (누락은 0토큰을 의미하지 않습니다)")

In [ ]:
def split_last_input(recorder):
    '''마지막 모델 호출에 전달된 입력을 구성 요소별 글자 수로 분해합니다. 토큰 수와 1:1로 일치하지는 않으며 어느 항목이 큰지 비교하는 용도입니다.'''
    parts = {"시스템 지침": 0, "도구 스키마": 0, "응답 스키마": 0, "대화 이력": 0, "도구 결과": 0}
    for message in recorder.last_messages:
        if isinstance(message, SystemMessage):
            parts["시스템 지침"] += len(message.text)
        elif isinstance(message, ToolMessage):
            parts["도구 결과"] += len(message.text)
        elif isinstance(message, HumanMessage | AIMessage):
            parts["대화 이력"] += len(message.text) + len(json.dumps(getattr(message, "tool_calls", []), ensure_ascii=False))
    parts["도구 스키마"] = len(json.dumps(recorder.last_params.get("tools", []), ensure_ascii=False))
    parts["응답 스키마"] = len(json.dumps(recorder.last_params.get("response_format", {}), ensure_ascii=False))
    return parts

# 마지막 호출의 입력 구성을 막대 그래프로 출력합니다.
parts = split_last_input(single["rec"])
print(bars(parts))
print("마지막 호출의 메시지 수:", len(single["rec"].last_messages), "| 모델 호출 0회 (이미 수집된 입력을 읽기만 합니다)")

**관찰할 것.** 호출별 표의 `입력 토큰` 열을 위에서 아래로 읽으며 앞서 예측한 배수와 비교해 보세요. `입력 토큰`은 매 호출마다 모델에 다시 전송하는 전체 입력 크기이고, `캐시 읽기`는 그중 프롬프트 캐시에서 읽어 온 분량입니다. 두 열을 나란히 비교하면 전체 입력이 증가하는 양과 매번 새로 처리하는 양이 다르다는 점을 확인할 수 있습니다. 따라서 비용을 분석할 때는 반드시 두 열을 함께 읽어야 합니다.

이어서 막대 분해 결과에서 어느 항목이 가장 큰지 찾아보세요. 만약 `도구 결과`가 가장 크다면, 이번 요청의 Context는 조회한 원문과 조사 데이터 때문에 커진 것입니다. 반면 `시스템 지침`이나 `도구 스키마`가 가장 크다면, 이는 조사 내용과 무관하게 매 호출마다 고정으로 붙는 기본 비용입니다. 에이전트를 나누어 Context를 분리했을 때 부모 에이전트 쪽에서 줄어드는 항목은 `도구 결과`입니다. 반대로 `시스템 지침`과 스키마 항목은 Worker를 나눌수록 각 Worker마다 하나씩 더 생겨납니다. 이 구분을 기억하며 다음 절의 결과를 예측해 보세요.

여기서 한 가지 사실을 정확히 짚고 넘어가야 합니다. 현재 실습 사례의 규모에서는 Single 구조라고 해서 Context 한도 초과로 실패하지는 않습니다. 위 호출표에서 우리가 확인한 것은 **한계의 징후(Context 누적 구조)**이지 실제 실패가 아닙니다. 결제 내역, 규정 원문, 가용률 조사 결과가 하나의 Context에 함께 쌓이고, 매 호출마다 그 전체를 반복해서 전송한다는 사실까지가 이 표를 근거로 말할 수 있는 전부입니다. 이 표만 보고 "Single 구조는 실패한다"라고 단정할 수는 없으며, 이 결과 역시 1회 실행 관찰입니다.

## 3. Supervisor: 결과를 돌려받는다

**문제.** 조사 업무를 여러 Worker로 나누면 Context가 실제로 분리될까요? 그리고 Context가 분리된다면 그 대가로 무엇을 치르게 될까요? Supervisor 구조에서는 부모 Agent가 `task` 도구를 호출하여 하위 Worker에게 작업을 위임하고, Worker의 **마지막 응답 메시지만** 돌려받아 부모가 최종 답변을 작성합니다. Deep Agents의 `subagents=` 설정이 바로 이 방식으로 동작합니다.

용어를 다시 한번 짚어 봅니다. 이 구조는 부모가 Worker를 도구처럼 호출하고 결과를 돌려받는다는 점에서 넓게는 Manager 패턴에 속합니다. 이 과정에서는 라이브러리 관례를 따라 **Supervisor**라고 부르며, 생성 함수 이름도 `build_supervisor`입니다. 여기서 다음에 어느 Worker를 호출할지 결정하는 주체는 고정된 그래프 간선이 아니라 부모 **모델**입니다. 동일한 제어 흐름에서 선택 주체를 모델 대신 일반 함수로 바꾼 뼈대는 5절에서 살펴봅니다.

**먼저 예측하세요.** 동일한 T-1001 요청을 Supervisor 구조로 실행하면, Single 구조와 비교했을 때 전체 모델 호출 횟수와 입력 토큰 합계는 늘어날까요, 줄어들까요? 또한 세 Worker(`billing`, `policy`, `outage`)의 실행 시간 구간은 서로 겹칠까요? 먼저 예상한 답을 적어 본 뒤 아래 셀들을 실행하여 확인합니다.

In [ ]:
# 세 명의 전문 Worker(billing, policy, outage) 사양과 build_supervisor의 소스 코드를 확인합니다.
preview_desk = make_desk(double_pay, label="preview")
specs = [subagent_spec(role, preview_desk, model) for role in ("billing", "policy", "outage")]
show_table([{"이름": s["name"], "설명": s["description"], "허용 도구": sorted(ROLE_TOOLS[s["name"]]), "시스템 지침 앞부분": s["system_prompt"][:60]} for s in specs])
print(inspect.getsource(build_supervisor))
print("모델 호출 0회. Worker 정의와 함수 소스를 확인하기만 합니다.")

In [ ]:
# 동시 위임 지침(concurrent_hint=True)을 포함한 Supervisor 에이전트를 실행합니다.
announce("Supervisor: 지침에 동시 위임 안내 포함", model_calls=14)
supervisor_desk = make_desk(double_pay, label="supervisor")
supervisor_app = build_supervisor(supervisor_desk, model, concurrent_hint=True)
supervisor_run = execute(supervisor_app, ask(double_pay["question"]), Recorder(supervisor_desk))
print("실행 오류:", supervisor_run["error"] or "없음", "| 시간(초):", supervisor_run["seconds"], "| outcome:", supervisor_run["final"].outcome if supervisor_run["final"] else None)
print("호출 주체 순서:", [c.get("agent") for c in supervisor_run["rec"].calls])

In [ ]:
# 부모(supervisor) 에이전트의 대화 메시지 목록과 task 위임 내역을 표로 확인합니다.
supervisor_messages = supervisor_run["state"].get("messages", [])
show_table(message_table(supervisor_messages))
show_table(task_rows(supervisor_messages))
print("부모 메시지 수:", len(supervisor_messages), "| 모델 호출 0회")

In [ ]:
# 각 Worker의 실행 시간 막대(timeline)와 동시 실행 겹침(overlap) 시간을 확인합니다.
print(timeline(supervisor_run["rec"].spans))
print("Worker 구간 겹침(초):", overlap_seconds(supervisor_run["rec"].spans))
print("Trace에 mas_lab이 남긴 구간:", supervisor_desk.trace.spans, "(delegation_spans가 task 호출마다 기록합니다. 위 Recorder의 구간과 비교해 보세요)")
print("모델 호출 0회")

In [ ]:
# 각 Worker가 내부에서 읽은 도구 결과 크기와 부모에게 반환한 메시지 크기를 비교합니다.
worker_tools = frozenset().union(*(ROLE_TOOLS[r] for r in ("billing", "policy", "outage")))
targets = {c["id"]: c["args"].get("subagent_type") for m in supervisor_messages for c in getattr(m, "tool_calls", []) if c["name"] == "task"}
returned = {targets[m.tool_call_id]: m.text for m in supervisor_messages if isinstance(m, ToolMessage) and m.tool_call_id in targets}
inside = supervisor_run["rec"].tool_results
show_table([{"Worker": w, "안에서 부른 도구 수": sum(t["agent"] == w for t in inside), "안에서 본 도구 결과 글자": sum(t["chars"] for t in inside if t["agent"] == w),
             "부모에게 돌아온 글자": len(text), "돌아온 글의 근거 참조 수": len(re.findall(r"F-[\w-]+|E-[0-9a-f]{12}", text))} for w, text in returned.items()])
seen = sorted({c["name"] for m in supervisor_messages for c in getattr(m, "tool_calls", [])} & worker_tools)
print("부모 대화에 나타난 Worker 전용 도구:", seen or "없음", "| 모델 호출 0회")

**관찰할 것.** 위 네 개의 출력 결과를 차례대로 살펴봅니다. 먼저 `task_rows` 표에서 한 번의 메시지에 `task 수`가 2 이상인 행이 있는지, 타임라인 그림에서 Worker 막대들이 세로로 겹치는지, 그리고 `Worker 구간 겹침(초)`가 0보다 큰지 확인해 보세요. 구간이 겹쳤다면 여러 Worker가 병렬로 실행된 것입니다. 그런데 여기서 **병렬로 실행하라고 결정한 주체는 그래프 코드가 아니라 부모 모델**입니다. 그래프 코드 어디에도 고정된 병렬 분기 간선은 없습니다. 따라서 만약 겹침이 0으로 나왔다면, 그것은 코드가 고장 난 것이 아니라 이번 실행에서 모델이 순차적으로 호출하기를 선택한 결과입니다.

네 번째 표는 Context 분리 효과를 구체적인 숫자로 보여 줍니다. `안에서 본 도구 결과 글자`와 `부모에게 돌아온 글자`를 비교해 보세요. 부모의 대화 이력에 Worker 전용 도구가 전혀 나타나지 않았다면, Worker가 조회한 긴 원문은 부모의 Context에 쌓이지 않은 것입니다. 이것이 Supervisor 구조의 핵심 이득이자 동시에 새로운 대가입니다. 즉, Worker가 요약 메시지에 담아 돌려주지 않은 사실은 부모 에이전트가 알 방법이 없습니다. `돌아온 글의 근거 참조 수`가 0인 Worker가 있는지도 확인해 보세요. 부모는 Worker가 반환한 텍스트에 적힌 `fact_ref`와 `evidence_id`만을 사용해 최종 인용을 작성할 수 있습니다. `subagent_spec`의 시스템 지침에서 근거(Evidence) 참조 번호를 본문에 반드시 포함하도록 요구하는 이유가 바로 여기에 있습니다.

Single 구조의 입력 토큰 합계와 비교할 때는 총 모델 호출 횟수와 호출당 평균 입력 크기를 나누어 살펴보세요. Supervisor는 호출당 입력 크기가 작아지더라도 전체 호출 횟수가 늘어날 수 있으므로, 총 토큰 합계가 어느 쪽으로 움직였는지는 이번 실행의 측정값을 통해 직접 확인해야 합니다.

아래 코드에서는 시스템 지침에서 "동시에 위임하라"는 문구만 제거했을 때 실행 구조가 어떻게 달라지는지 확인합니다.

In [ ]:
# 실패 재현: 지침에서 "동시에"라는 문구를 빼면 병렬 실행 구조가 그대로 유지되는지 확인합니다.
announce("Supervisor: 동시 위임 지침 제외", model_calls=14)
plain_desk = make_desk(double_pay, label="supervisor-plain")
plain_app = build_supervisor(plain_desk, model, concurrent_hint=False)
plain = execute(plain_app, ask(double_pay["question"]), Recorder(plain_desk))
print("실행 오류:", plain["error"] or "없음", "| 시간(초):", plain["seconds"], "| outcome:", plain["final"].outcome if plain["final"] else None)
print("호출 주체 순서:", [c.get("agent") for c in plain["rec"].calls])

In [ ]:
def delegation_row(name, run):
    # 위임 지침 유무에 따른 실행 순서와 겹침 시간을 한 행으로 요약합니다.
    groups, totals = task_rows(run["state"].get("messages", [])), run["rec"].totals()
    return {"조건": name, "위임 순서": " → ".join(f"[{g['위임 대상']}]" for g in groups), "메시지당 task 수": [g["task 수"] for g in groups],
            "시간(초)": run["seconds"], "겹침(초)": overlap_seconds(run["rec"].spans), "모델 호출": totals["model_calls"],
            "입력 토큰": totals["input_tokens"], "캐시 읽기": totals["cache_read_tokens"], "outcome": run["final"].outcome if run["final"] else None}

# 동시 위임 지침이 있을 때와 없을 때의 실행 결과를 비교합니다.
show_table([delegation_row("동시 위임 지침 있음", supervisor_run), delegation_row("지침 없음", plain)])
print(timeline(plain["rec"].spans))
print("모델 호출 0회. 각 행은 1회 실행 관찰 결과입니다.")

**관찰할 것.** 위 표에서 두 실행의 `메시지당 task 수`와 `겹침(초)`를 비교해 보세요. 두 실행의 그래프 코드는 완전히 동일하며, 단지 시스템 지침의 문구 한 줄만 다릅니다. 만약 지침을 뺐더니 Worker 위임이 메시지마다 하나씩 순차 실행으로 바뀌었다면, 프롬프트 한 줄이 실제 실행 아키텍처를 바꾼 셈입니다. 반대로 지침을 뺐는데도 한꺼번에 위임했다면, 그것은 이번 1회 실행에서 모델이 우연히 그렇게 선택한 것일 뿐 지침 없이도 병렬이 보장된다는 증거가 아닙니다. 어느 쪽 결과가 나왔든 핵심 결론은 동일합니다. **실행 구조가 코드에 의해 고정되어 있지 않다는 점**입니다. 다음 노트북(03a)에서는 동일한 병렬 실행을 프롬프트가 아니라 그래프 코드로 고정하는 방법을 다룹니다.

또한 두 행의 `시간(초)` 차이를 곧바로 병렬화의 순수 효과라고 단정하지 않도록 주의하세요. 두 실행은 위임 설명의 길이와 내부 모델 호출 횟수까지 달라졌을 수 있기 때문입니다. 작업 분해를 완전히 고정한 상태에서 동시성 스위치 하나로만 시간을 비교하는 통제 실험은 03a와 06 노트북에서 수행합니다.

아래 코드에서는 예산(budget) 및 도구 허용 목록 middleware가 부모와 Worker에 각각 어떻게 적용되는지 확인합니다.

In [ ]:
# 부모와 Worker 모두에 middleware를 걸었을 때와 부모에만 걸었을 때 예산 집계 차이를 계산합니다.
report = supervisor_desk.budget.report()["counts"]
calls, totals = supervisor_run["rec"].calls, supervisor_run["rec"].totals()
parent_calls = sum(c.get("agent") == "supervisor" for c in calls)
parent_tools = sum(e["event"] == "tool" and e["name"] in ROLE_TOOLS["writer"] | {"task"} for e in supervisor_desk.budget.report()["events"])
show_table([{"middleware 설정": "부모와 Worker 모두 (실제 실행)", "예산이 센 모델 호출": report.get("model", 0), "예산이 센 도구 호출": report.get("tool", 0),
             "실제 모델 호출": totals["model_calls"], "예산 밖 모델 호출": totals["model_calls"] - report.get("model", 0), "Worker 허용 도구 검사": "적용"},
            {"middleware 설정": "부모만 적용 (동일 실행 가정 계산)", "예산이 센 모델 호출": parent_calls, "예산이 센 도구 호출": parent_tools,
             "실제 모델 호출": totals["model_calls"], "예산 밖 모델 호출": totals["model_calls"] - parent_calls, "Worker 허용 도구 검사": "없음"}])
task_tool = next((t for t in supervisor_run["rec"].last_params.get("tools", []) if t.get("name") == "task"), {})
print("이번 실행의 task 도구 설명에 general-purpose가 포함되어 있는가:", "general-purpose" in task_tool.get("description", ""))
print("예산 집계 합계:", report, "| 모델 호출 0회")

**관찰할 것.** `subagent_spec` 함수는 각 Worker를 생성할 때마다 `allowlist_middleware(desk.budget, ...)`를 개별적으로 등록합니다. 이때 부모 에이전트와 동일한 `Budget` 인스턴스를 공유하므로 모든 Worker의 호출이 하나의 예산(budget)에 합산됩니다. 첫 번째 행의 `예산 밖 모델 호출`이 0이라면, 실제로 발생한 모든 모델 호출이 예산 통제 안에 있었다는 뜻입니다. 두 번째 행은 동일한 실행에서 만약 "부모에게만 middleware를 걸었다면" 어떻게 되었을지 계산한 결과입니다. Deep Agents는 부모의 사용자 정의 middleware를 하위 subagent에 자동으로 상속하지 않으므로, 부모에만 설정하면 Worker 내부의 모델 호출은 예산 집계에서 누락되고 도구 접근 제한도 풀리게 됩니다.

출력 마지막 부분의 `이번 실행의 task 도구 설명에 general-purpose가 포함되어 있는가` 항목도 함께 확인해 보세요. 이 값은 부모에게 전달된 `task` 도구의 설명 문자열에서 `general-purpose`라는 Worker 이름을 검색한 결과입니다. Deep Agents가 기본적으로 자동 추가하는 `general-purpose` Worker에는 우리가 만든 예산 및 허용 도구 middleware가 걸려 있지 않기 때문에, 만약 부모가 그 Worker에게 작업을 위임했다면 첫 번째 행의 `예산 밖 모델 호출`이 0보다 커졌을 것입니다. 이 자동 추가 Worker가 무엇이고 우리가 직접 설계한 Worker와 어떻게 다른지 다음 소절에서 모델을 호출하지 않고 코드 객체로 자세히 비교해 봅니다.

### 자동으로 끼워 넣은 Worker와 직접 설계한 Worker

앞서 살펴본 `task` 도구의 설명(description)은 부모 모델이 작업을 위임할 수 있는 후보 Worker 목록이기도 합니다. 우리가 `subagent_spec`으로 직접 설계한 Worker는 `billing`, `policy`, `outage` 세 개뿐입니다. 그런데 Deep Agents 프레임워크는 우리가 별도로 지정하지 않아도 `general-purpose`라는 범용 Worker를 기본으로 하나 더 추가합니다. 똑같이 Worker라는 이름을 쓰지만 이를 설계한 주체가 다릅니다. 세 전문 Worker는 이번 업무 사례에 맞춰 조회 가능한 도구와 예산(budget) 한도를 우리가 직접 통제했지만, `general-purpose` Worker는 프레임워크의 기본 설정을 그대로 따릅니다.

**먼저 예측하세요.** 프레임워크가 자동으로 끼워 넣은 `general-purpose` Worker는 이번 T-1001 사례에서 어떤 정보를 볼 수 있고 어떤 도구를 사용할 수 있을까요? 부모 에이전트의 도구를 물려받을까요, 아니면 우리가 만든 허용 목록과 예산을 적용받을까요? 결제 내역 조회 도구는 가지고 있을까요? 예상한 내용을 적어 본 뒤 아래 두 코드 셀을 실행하여 확인합니다. 두 셀은 에이전트 객체만 생성하여 내부 속성을 검사할 뿐 모델을 호출하지 않습니다.

In [ ]:
# 동일한 build_supervisor를 두 가지 설정으로 각각 생성하여, 부모의 task 도구가 노출하는 Worker 목록을 비교합니다.
# 에이전트 그래프 객체를 생성하는 것은 모델 호출이 아닙니다. 아래 desk와 앱 객체는 이 셀에서만 사용합니다.

# 1) 프레임워크 기본 설정: 컨텍스트 블록 안에서만 general-purpose 자동 추가를 활성화합니다(비교 전용).
default_desk = make_desk(double_pay, label="framework-default")
with framework_default_subagent(model):
    default_app = build_supervisor(default_desk, model)
default_targets = task_targets(default_app)

# 2) 이 과정의 기본 설정: mas_lab을 import할 때 이미 general-purpose를 비활성화했으므로 그대로 생성합니다.
course_desk = make_desk(double_pay, label="course-setting")
course_app = build_supervisor(course_desk, model)
course_targets = task_targets(course_app)

show_table([
    {"설정": "프레임워크 기본", "task가 내놓는 Worker": ", ".join(default_targets), "개수": len(default_targets),
     "general-purpose 포함": "general-purpose" in default_targets},
    {"설정": "이 과정의 설정", "task가 내놓는 Worker": ", ".join(course_targets), "개수": len(course_targets),
     "general-purpose 포함": "general-purpose" in course_targets},
])
print("읽은 위치: mas_lab/harness.py의 task_targets (task 도구 설명 내 Worker 목록 구간) | 모델 호출 0회")

In [ ]:
# 우리가 직접 설계한 세 Worker와 프레임워크가 자동 추가한 general-purpose를 동일한 기준으로 비교합니다.
comparison_desk = make_desk(double_pay, label="worker-comparison")
parent_tool_names = sorted(t.name for t in comparison_desk.tools("writer"))        # build_supervisor가 부모에게 제공하는 도구 목록
lookup_tool_names = sorted(set().union(*(ROLE_TOOLS[role] for role in ("billing", "policy", "outage"))))  # 이번 사례의 전체 조회 도구 목록


def allowlist_of(spec: dict):
    """Worker 사양의 첫 번째 middleware(bounded_model) 클로저에서 허용 도구 목록과 연결된 Budget 객체를 추출합니다."""
    bounded_model = spec["middleware"][0].wrap_model_call.__closure__[0].cell_contents
    bound = inspect.getclosurevars(bounded_model).nonlocals
    return sorted(bound["allowed"]), bound["budget"] is comparison_desk.budget


rows = []
for role in ("billing", "policy", "outage"):
    spec = subagent_spec(role, comparison_desk, model)                              # 1) 직접 설계한 Worker의 사양
    allowed, shares_budget = allowlist_of(spec)                                      # 2) 허용 도구 목록과 예산 공유 여부 확인
    rows.append({"이름": spec["name"], "설명(누가 언제 부르라고 했나)": spec["description"],
                 "보이는 도구": ", ".join(t.name for t in spec["tools"]),
                 "예산·허용 목록": f"허용 {allowed}, 부모와 같은 Budget: {shares_budget}",
                 "지침이 근거 번호를 요구하는가": "fact_ref" in spec["system_prompt"] and "evidence_id" in spec["system_prompt"],
                 "읽은 곳": "mas_lab/workers.py:173, :25"})

gp = GENERAL_PURPOSE_SUBAGENT                                                         # 3) 프레임워크가 정의한 기본 범용 Worker 사양
rows.append({"이름": gp["name"], "설명(누가 언제 부르라고 했나)": gp["description"],
             "보이는 도구": f"부모의 도구 {parent_tool_names} + 파일 도구",
             "예산·허용 목록": "없음 (부모의 사용자 정의 middleware를 물려받지 않음)",
             "지침이 근거 번호를 요구하는가": "fact_ref" in gp["system_prompt"],
             "읽은 곳": "deepagents/graph.py:880-884, :868-871, deepagents/middleware/subagents.py:504"})
show_table(rows)

# 4) 이번 사례의 조회 도구를 general-purpose가 가지고 있는지 교집합으로 확인합니다.
print("이번 사례의 조회 도구:", lookup_tool_names)
print("general-purpose가 가진 조회 도구:", sorted(set(parent_tool_names) & set(lookup_tool_names)) or "없음")
print("모델 호출 0회")

**관찰할 것.** 위 비교표에서 `general-purpose` 행을 우리가 직접 설계한 세 Worker 행과 항목별로 대조해 보세요. `general-purpose` Worker의 `보이는 도구`는 부모 에이전트가 가진 도구와 기본 파일 도구입니다. 그렇다면 이 Worker는 부모와 동일한 도구를 바라보므로, 에이전트를 분리해서 얻는 권한 및 Context 경계가 존재할까요? 또한 이번 사례를 해결하는 데 필요한 결제·규정·가용률 조회 도구를 가지고 있나요? 조회 도구가 전혀 없다면 이 Worker가 실제로 수행할 수 있는 조사 업무는 무엇일까요? 나아가 `예산·허용 목록` 항목이 비어 있다면 이 Worker가 내부에서 호출하는 모델 비용은 어느 예산(budget) 객체가 집계할까요?

이 질문들은 이 과정에서 세운 핵심 설계 원칙으로 다시 연결됩니다.

- 04 노트북 4절: 예산(budget)은 모델 외부에서 단일 기준으로 집계합니다. 예산 통제 밖에서 돌아가는 Worker가 섞이면 전체 비용을 하나로 관리할 수 없습니다.
- 05 노트북 3절: 에이전트를 나누는 기준은 **무엇을 볼 수 있고 무엇을 사용할 수 있는가**입니다. 부모와 똑같은 도구를 보는 Worker는 권한 경계가 없습니다.
- 05 노트북 4절: SubAgent 분리는 Context가 격리되거나 허용 도구가 다를 때만 정당화됩니다. `general-purpose`의 도구는 부모의 도구와 동일합니다.
- 05 노트북 7절: 공정한 아키텍처 비교 실험은 모든 구조가 동일한 예산 한도를 공유한다는 가정 위에 서 있습니다. 예산 밖에서 동작하는 Worker가 끼어들면 이 비교 가정이 무너집니다.

핵심 문제는 `general-purpose` Worker 자체가 잘못 만들어졌다는 것이 아닙니다. **우리 업무 사례에 맞는 권한과 예산 경계를 설계한 주체가 없다**는 점이 문제입니다. 따라서 이 과정에서는 `mas_lab` 패키지를 불러올 때(`mas_lab/__init__.py`) 모든 Deep Agent에서 이 자동 추가 Worker를 비활성화합니다. 바로 앞 셀 비교표의 `이 과정의 설정` 행이 그 결과입니다.

물론 옵션을 끄는 것만으로 모든 보안과 통제가 끝나지는 않습니다.1회 실행에서 호출되지 않았다는 사실이 곧 호출이 차단되어 있다는 증거는 아니므로, 권한 경계는 프롬프트가 아니라 코드와 설정으로 검증해야 합니다. 이 주제는 Day08 가드레일 실습에서 더 깊이 다룹니다.

## 4. Handoff: 대화를 넘긴다

**문제.** Supervisor 구조에서는 언제나 부모 에이전트가 모든 결과를 모아 최종 응답을 작성합니다. 그런데 고객과의 대화가 단순 결제 문의에서 시작해 기술적인 장애 보상 문의로 넘어간다면, 결제 담당자가 끝까지 중간에서 말을 전달만 하는 것이 자연스러울까요? 해당 분야의 담당자(owner)가 대화 제어권을 직접 넘겨받아 이어서 응답하면 중간 전달 단계가 줄어듭니다. 하지만 그 대가로 이전 담당자가 알고 있던 사실은 **인계 메모(`notes`)에 적어 넘긴 만큼만** 다음 담당자에게 전달됩니다.

여기서 혼동하기 쉬운 두 용어를 명확히 구분합니다. **Network는 연결 모양(토폴로지)**입니다. 여러 노드가 서로를 호출하거나 이동할 수 있도록 연결되어 있다는 뜻입니다. 반면 **Handoff는 제어권 이전 규칙**입니다. 현재 누가 대화의 담당자(owner)이고, 어떤 조건에서 제어권을 넘기며, 한 번 넘기면 결과를 다시 돌려받지 않는다는 운영 규칙을 뜻합니다. 연결 모양(Network)만 만들어 두고 제어권 규칙(Handoff 종료 조건)을 세우지 않으면, 두 담당자가 서로 책임을 미루며 무한히 대화를 넘기게 됩니다.

**먼저 예측하세요.** 명확한 종료 조건 없이 서로에게 대화를 넘기기만 하는 두 담당자 노드를 실행하면 어떻게 될까요? 스스로 멈출까요, 아니면 예외가 발생할까요? 만약 멈춘다면 노드가 몇 번 실행된 뒤에 멈출까요? `recursion_limit=8` 설정을 염두에 두고 예상 결과를 적은 뒤 아래 셀을 실행하여 확인합니다.

In [ ]:
# 실패 재현: 종료 조건 없이 서로에게 제어권을 넘기는 두 담당자 노드를 실행합니다.
class BounceState(TypedDict, total=False):
    bounces: int
    path: list[str]
    status: str

def build_bounce(limit=None, visits=None):
    '''일반 함수 노드 두 개가 Command(goto=...)를 사용해 서로에게 제어권을 넘깁니다. limit이 지정되면 해당 횟수에서 멈추고 사람에게 확인을 요청합니다.'''
    def owner(me, other):
        def node(state) -> Command[Literal["billing", "technical", "__end__"]]:
            bounces, path = state.get("bounces", 0), [*state.get("path", []), me]
            if visits is not None:
                visits.append(me)
            # 왕복 상한(limit)에 도달하면 무한 루프를 멈추고 상태를 needs_clarification으로 기록합니다.
            if limit is not None and bounces >= limit:
                return Command(update={"path": path, "status": "needs_clarification"}, goto=END)
            return Command(update={"path": path, "bounces": bounces + 1, "status": "running"}, goto=other)
        return node
    graph = StateGraph(BounceState)
    graph.add_node("billing", owner("billing", "technical"))
    graph.add_node("technical", owner("technical", "billing"))
    graph.add_edge(START, "billing")
    return graph.compile()

visits = []
try:
    build_bounce(visits=visits).invoke({}, config={"recursion_limit": 8})
except GraphRecursionError as error:
    print(type(error).__name__, "|", str(error).split(".")[0])
print("실행된 노드 수:", len(visits), "| 실행 순서:", visits, "| 모델 호출 0회")

In [ ]:
# 상태에 왕복 상한(LIMIT = 2)을 적용하여 예외 대신 정상적인 확인 요청 상태로 종료되는지 확인합니다.
LIMIT = 2  # MAX_HANDOFFS와 동일한 값으로 설정합니다.
result = build_bounce(limit=LIMIT).invoke({}, config={"recursion_limit": 25})
show_table([{"왕복 상한": LIMIT, "종료 상태": result["status"], "넘긴 횟수": result["bounces"], "경로": " → ".join(result["path"])}])
print("build_handoff의 기본 상한 MAX_HANDOFFS:", MAX_HANDOFFS, "| 모델 호출 0회")

In [ ]:
# 실패 재현: Command 반환 타입 힌트를 적지 않으면 그래프 시각화에서 이동 경로가 누락되는 현상을 확인합니다.
def plain_a(state):
    return Command(goto="b")
def plain_b(state):
    return Command(goto=END)
def typed_a(state) -> Command[Literal["b"]]:
    return Command(goto="b")
def typed_b(state) -> Command[Literal["a", "__end__"]]:
    return Command(goto=END)

def edges_of(a, b):
    # 두 노드로 구성된 그래프를 컴파일하고 추출된 간선(edges) 목록을 반환합니다.
    graph = StateGraph(BounceState)
    graph.add_node("a", a)
    graph.add_node("b", b)
    graph.add_edge(START, "a")
    return [{"출발": e.source, "도착": e.target, "조건부": e.conditional} for e in graph.compile().get_graph().edges]

for name, a, b in (("반환 타입 힌트 없음", plain_a, plain_b), ("Command[Literal[...]] 명시", typed_a, typed_b)):
    print(name)
    show_table(edges_of(a, b))
print("모델 호출 0회")

**관찰할 것.** 첫 번째 실패 재현에서 발생한 예외 이름(`GraphRecursionError`)과 `실행된 노드 수`를 확인해 보세요. 그래프 런타임은 스스로 "이 담당자 간 왕복이 무의미하다"라고 판단하지 못합니다. `recursion_limit`은 최후의 안전장치일 뿐이며, 이 한도에 도달하면 예외가 발생해 실행이 끊기므로 고객에게 전달할 정상적인 응답이 남지 않습니다. 반면 왕복 상한(`LIMIT`)을 상태 조건으로 명시한 두 번째 표에서는 `종료 상태`가 `needs_clarification`으로 기록되고 이동 `경로`도 온전히 남습니다. 똑같이 실행을 멈추더라도 예외로 비정상 종료되는 것과 사람에게 인계할 상태를 남기고 종료되는 것은 후속 운영 조치가 완전히 다릅니다.

세 번째 셀의 간선 표 두 개도 비교해 보세요. 노드 함수에 `Command[Literal[...]]` 반환 타입 힌트를 적지 않으면 `a`에서 `b`로 이동하는 간선이 그래프 구조 정보에서 사라집니다. 실행 자체는 동작하더라도, 그래프 다이어그램을 읽는 동료나 디버깅 도구는 이 그래프가 `a`에서 곧바로 끝난다고 오해하게 됩니다. 그래프는 실행 코드인 동시에 시스템 구조를 설명하는 문서이므로, `build_handoff`의 모든 노드에는 `Command[Literal[...]]` 타입이 명시되어 있습니다.

아래 코드에서는 실제 모델이 동작하는 `build_handoff`의 구현 코드와 그래프 구조를 출력합니다.

In [ ]:
# build_handoff의 담당자별 도구 설정(OWNER_TOOLS)과 그래프 구조를 확인합니다.
source = inspect.getsource(inspect.getmodule(build_handoff))
print(source[source.index("OWNER_TOOLS ="):])
display_desk = make_desk(followup, label="preview")
show_graph(build_handoff(display_desk, model))
print("모델 호출 0회. 그래프 객체만 생성하며, 내부 Agent는 노드가 실행될 때 만들어집니다.")

위 코드와 그래프에서 세 가지 핵심 구조를 먼저 살펴보세요.
1. `resume` 노드가 진입점(Triage) 역할을 합니다. 새로운 대화 턴이 시작될 때 기존 담당자(owner)를 그대로 유지할지, 다른 담당자로 바꿀지 결정합니다.
2. `billing`과 `technical` 노드가 실제 대화를 처리하는 담당자입니다.
3. `OWNER_TOOLS` 정의를 보면 `technical` 담당자에게는 결제 조회 도구(`lookup_payments`)가 없습니다. 따라서 `technical` 담당자가 고객의 결제 내역을 알 수 있는 유일한 통로는 앞선 담당자가 `notes` 필드로 넘겨준 인계 메모뿐입니다.

규정상 서비스 크레딧은 총결제 금액이 아니라 **기본 이용료분**에 보상 비율을 곱해 계산해야 합니다(`data/support/billing.json`의 P-9002 결제는 총액 1,200,000원이며, 그중 기본 이용료분이 1,000,000원, 사용량 초과분이 200,000원입니다). 그런데 이 기본 이용료분 수치가 인계 메모에 포함될지 여부는 코드가 강제하는 것이 아니라 앞선 `billing` 담당자 모델의 요약 판단에 달려 있습니다.

**먼저 예측하세요.** 첫 번째 턴에서 `billing` 담당자가 `technical` 담당자에게 넘기는 메모(`notes`)에 '기본 이용료분 1,000,000원'이라는 정보가 정확히 포함될까요? 만약 포함되지 않는다면 `technical` 담당자는 크레딧을 어떻게 처리할까요? 결제 총액(1,200,000원)으로 잘못 계산할까요, 아니면 계산을 보류할까요? 예상한 시나리오를 적은 뒤 아래 셀들을 실행하여 확인합니다.

In [ ]:
# Handoff 구조로 첫 번째 턴(T-1001 복합 문의)을 실행합니다.
announce("Handoff 턴 1: T-1001 초기 문의 처리", model_calls=12)
handoff_desk = make_desk(followup, label="handoff")
handoff_app = build_handoff(handoff_desk, model)  # 내부에 체크포인터(checkpointer)가 포함되어 있어 동일한 app과 thread_id를 쓰면 상태가 이어집니다.
thread_id = uuid4().hex

def handoff_turn(question):
    # 한 턴의 실행 전후로 추가된 trace 이벤트와 예산 이벤트를 분리하여 기록합니다.
    events_before, budget_before = len(handoff_desk.trace.events), len(handoff_desk.budget.events)
    run = execute(handoff_app, {"question": question}, Recorder(handoff_desk), thread_id)
    return {**run, "question": question, "events": handoff_desk.trace.events[events_before:], "budget_events": handoff_desk.budget.events[budget_before:]}

turn1 = handoff_turn(followup["turns"][0])
print("실행 오류:", turn1["error"] or "없음", "| 시간(초):", turn1["seconds"], "| 모델 호출:", turn1["rec"].totals()["model_calls"])

In [ ]:
# 턴 1에서 발생한 담당자 제어권 이전 사건과 인계 메모(notes), 크레딧 계산 기준을 확인합니다.
CONTROL = {"handoff", "owner_retained", "handoff_limit", "final_answer"}

def control_rows(events):
    # 제어권 관련 이벤트만 추출하여 시각, 주체, 이전/새 담당자, 사유를 표로 정리합니다.
    return [{"시각": e["at"], "사건": e["kind"], "주체": e["actor"], "이전 담당자": e.get("previous_owner"), "새 담당자": e.get("owner"), "이유": (e.get("reason") or "")[:60]}
            for e in events if e["kind"] in CONTROL]

show_table(control_rows(turn1["events"]))
state = turn1["state"]
print("active owner:", state.get("owner"), "| 턴 내 전환 횟수:", state.get("handoffs"), "| 메모 글자 수:", len(state.get("notes") or ""))
print("넘어간 메모:
", state.get("notes") or "(없음)")
print("
최종 응답:
", turn1["final"].text if turn1["final"] else "(없음)")
notes = state.get("notes") or ""
show_table([{"크레딧 계산 기준": credit_basis(handoff_desk, turn1["events"]), "답의 금액": sorted(set(amounts_in(turn1["final"]))),
             "메모에 '기본 이용료' 언급": "기본 이용료" in notes, "메모에 1,000,000 있음": "1,000,000" in notes or "1000000" in notes}])
print("모델 호출 0회")

In [ ]:
# 동일한 thread_id에서 후속 질문(턴 2)을 이어서 실행합니다.
announce("Handoff 턴 2: 동일 thread의 후속 질문 처리", model_calls=8)
handoff_desk.budget.started = time.monotonic()  # 시간 예산은 턴 단위로 초기화합니다. 앞 턴의 실행 시간과 표를 읽은 시간은 제외하며, 호출 횟수는 누적해서 집계합니다.
turn2 = handoff_turn(followup["turns"][1])
print("실행 오류:", turn2["error"] or "없음", "| 시간(초):", turn2["seconds"], "| 모델 호출:", turn2["rec"].totals()["model_calls"])
print("후속 질문:", turn2["question"])
print("최종 응답:
", turn2["final"].text if turn2["final"] else "(없음)")

In [ ]:
def turn_row(index, turn):
    # 각 대화 턴의 담당자 변화, 최종 응답자, 검색/읽기 횟수, 크레딧 계산 기준을 한 행으로 요약합니다.
    final, totals = turn["final"], turn["rec"].totals()
    owners = [f"{e.get('previous_owner')} → {e['owner']} ({e['kind']})" for e in turn["events"] if e["kind"] in {"handoff", "owner_retained"}]
    gated = gate_answer(final, handoff_desk, request_id=thread_id) if final else None
    return {"턴": index, "담당자 사건": owners, "최종 응답자": next((e["actor"] for e in turn["events"] if e["kind"] == "final_answer"), None),
            "검색": sum(e["event"] == "search" for e in turn["budget_events"]), "원문 읽기": sum(e["event"] == "read" for e in turn["budget_events"]),
            "모델 호출": totals["model_calls"], "입력 토큰": totals["input_tokens"], "시간(초)": turn["seconds"],
            "담당자 outcome": final.outcome if final else None, "게이트 뒤 outcome": gated.outcome if gated else None,
            "크레딧 계산 기준": credit_basis(handoff_desk, turn["events"]), "인용 수": len(final.evidence_refs) if final else 0}

# 턴 1과 턴 2의 실행 결과를 나란히 비교합니다.
show_table([turn_row(1, turn1), turn_row(2, turn2)])
print("원장(ledger) 기록 건수:", len(handoff_desk.ledger.refunds), "| 예산 누적:", handoff_desk.budget.report()["counts"], "| 모델 호출 0회")

**관찰할 것.** 먼저 `크레딧 계산 기준` 항목을 확인해 보세요. 이 항목은 `calculate_credit` 도구에 실제로 전달된 인자를 결제 원장의 기본 이용료분(1,000,000원)과 대조한 결과이며, 이번 Handoff 실험의 핵심 관찰 지점입니다. 이번 실행에서 나온 값에 따라 해석이 다음과 같이 달라집니다.

- **`기본 이용료분과 일치`**: 이번 1회 실행에서는 인계 메모(`notes`)에 필요한 세부 금액이 잘 포함되었습니다. 하지만 이것이 구조적으로 안전하다는 증거는 아닙니다. 어떤 수치를 메모에 적을지 결정한 주체는 코드가 아니라 모델이므로, 다음 실행에서는 누락될 수 있습니다.
- **`기본 이용료분 1,000,000과 다름`**: 담당자 간 정보 전달 손실이 잘못된 크레딧 계산으로 직접 드러난 경우입니다. `technical` 담당자는 결제 원장을 직접 조회할 도구가 없으므로, 메모에 적힌 결제 총액(1,200,000원)을 계산 기준으로 삼았을 가능성이 높습니다. 위에 출력된 메모 본문을 읽고 어디서 정보가 빠졌는지 찾아보세요.
- **`계산 도구 호출 없음`**: `calculate_credit` 도구를 호출하지 않은 경우입니다. `답의 금액`과 최종 응답 본문에 크레딧 금액이 없다면, 기준 금액을 확정하지 못해 계산을 보류한 것입니다. 틀린 금액을 안내하지는 않았지만 고객의 질문에 끝까지 답하지도 못한 상태입니다. 응답 본문에서 왜 계산하지 못했다고 설명하는지 읽고, 표의 `메모에 '기본 이용료' 언급` 항목과 대조해 보세요. 설령 `메모에 1,000,000 있음`이 `True`이더라도 안심할 수 없습니다. 다른 청구서(예: P-9000)의 결제 금액일 수 있으므로 출력된 메모 원문에서 어떤 항목의 금액인지 직접 확인해야 합니다. 만약 도구를 호출하지 않았는데도 최종 답변에 크레딧 금액이 적혀 있다면 모델이 암산으로 지어낸 값이므로 더욱 주의해야 합니다.

어느 결과가 나왔든 동일한 질문을 던져 봐야 합니다. **담당자(owner)가 바뀔 때 이 필수 수치가 누락되지 않도록 보장하는 장치가 코드에 있는가?** 최종 응답 스키마(`FinalAnswer`) 검사를 통과했더라도 `calculate_credit`에 들어간 입력값이 맞았는지까지는 스키마가 알려 주지 않습니다. 또한 표의 `담당자 사건` 열도 함께 살펴보세요. 턴 2에서 `owner_retained`(담당자 유지)가 나타났다면 담당자 전환 비용을 아낀 것입니다. 다만 후속 질문의 주제가 바뀌었는데도 기존 담당자를 잘못 유지하면 필요한 도구가 없는 담당자가 억지로 답변하게 됩니다. 반대로 `handoff`가 발생했다면 다시 메모를 요약해 넘기는 비용을 치른 것입니다.

`검색`과 `원문 읽기` 열에서 턴 2가 규정 원문을 새로 검색했는지도 확인해 보세요. 이전 턴의 대화 내역은 맥락을 파악하는 단서일 뿐 그 자체로 근거(Evidence)가 되지는 않습니다. 다시 검색했다면 비용을 들여 근거를 새로 확보한 것이고, 검색하지 않았다면 턴 2 응답의 인용 근거가 무엇인지 `인용 수` 열과 함께 점검해야 합니다.

마지막으로 `담당자 outcome`과 `게이트 뒤 outcome`이 서로 다를 수 있다는 점에 주목하세요. 환불 제안(`proposal`)은 한 턴 안에서 담당자가 바뀌어도 `HandoffState.proposal`에 유지되며, 코드 게이트(`gate_answer`)가 그 제안을 바탕으로 최종 허용·승인·거절을 다시 판정합니다. 반면 새로운 고객 턴을 여는 Triage(`resume`)는 이전 턴의 제안을 비웁니다. 덕분에 턴 2의 질문이 단순한 신청 기한 문의일 때 턴 1의 환불 제안 때문에 턴 2의 outcome까지 왜곡되는 일을 막을 수 있습니다. 물론 여기에도 대가가 따릅니다. 턴 1에서 승인 대기 중이던 환불 제안이 턴 2의 답변으로 자동 승계되지 않는다는 점입니다. 위 비교표는 각 턴마다 `gate_answer`를 호출해 보여 주므로 턴 1 행에서 게이트 판정을 확인할 수 있지만, 일괄 평가 함수인 `run_case`는 마지막 턴만 게이트에 통과시키므로 앞선 턴의 미결 제안을 실행하지 않고 `earlier_turn_proposal_ungated` 이벤트로만 기록합니다. 즉, 현재 턴의 outcome이 현재 질문에 대한 판정을 정확히 나타내게 된 대신, 이전 턴의 승인 대기 제안은 운영자가 별도로 챙겨야 합니다. 아직 승인이 이루어지지 않았으므로 `원장(ledger) 기록 건수`는 0이어야 정상입니다.

## 5. Supervisor, Orchestrator-Worker, Hierarchical, Plan-and-Execute 읽기

**문제.** Supervisor 구조는 3절에서 실제 모델로 실행해 보았습니다. 그렇다면 그 제어 흐름을 일반 함수 뼈대로만 추려내면 어떤 모양일까요? 그리고 이번 노트북에서 실제 모델로 돌리지 않은 나머지 구조들(Hierarchical, Plan-and-Execute)은 어떤 문제를 해결할 때 사용할까요?

이 절의 코드 셀들은 모델을 전혀 호출하지 않습니다. 아키텍처 간의 본질적인 차이는 **제어 흐름과 상태 전달의 차이**이므로 일반 함수 노드로도 명확하게 드러납니다. 모델 호출 없이 일반 함수로 실행하는 것은 임시방편이 아니라 제어 구조 자체를 가장 투명하게 관찰하는 방법입니다. 따라서 일반 함수 노드에 가짜 모델 응답 문장을 넣지 않았으며, 실제 모델을 결합한 구현은 03과 04 노트북에서 이어서 다룹니다.

먼저 Supervisor의 제어 흐름을 일반 함수 뼈대로 살펴봅니다. 3절에서는 부모 모델이 `task` 도구로 "다음에 실행할 Worker"를 골랐다면, 아래 코드 뼈대에서는 `route` 함수가 남은 작업 목록(`pending`)을 보고 다음 Worker를 고릅니다. 선택하는 주체가 모델이든 일반 함수든 그래프의 제어 흐름 모양은 동일합니다.

한편 Orchestrator-Worker 구조는 03a 노트북의 병렬 절에서 실제 모델로 실행합니다. 여기서는 Supervisor와의 위임 방식 차이만 짚어 둡니다. Supervisor는 "다음 작업 하나"를 고르고 그 결과를 확인한 뒤 다시 다음 작업을 고릅니다. 반면 Orchestrator-Worker는 처음에 "필요한 작업 전체"를 나누어 한꺼번에 내보낸 뒤 결과를 모읍니다. 아래 Supervisor 뼈대에서 `pending` 목록을 한 번에 모두 내보내도록 바꾸면 바로 Orchestrator-Worker의 모양이 됩니다.

아래 코드는 Supervisor 뼈대, Hierarchical(계층형 보고), Plan-and-Execute(고정 계획 대 재계획)를 차례대로 구성하고 실행합니다.

In [ ]:
# Supervisor 제어 흐름을 일반 함수 노드로 구현하여 실패 시 재시도 루프가 어떻게 도는지 확인합니다.
class SupervisorState(TypedDict, total=False):
    pending: list[str]
    attempts: dict[str, int]
    order: Annotated[list[str], add]

def choose(state):
    # Supervisor 노드가 실행되었음을 순서 기록(order)에 남깁니다.
    return {"order": ["supervisor"]}

def route(state) -> Literal["billing", "policy", "__end__"]:
    # 남은 작업(pending)이 있으면 첫 번째 작업을 고르고, 없으면 종료(END)합니다.
    return state["pending"][0] if state["pending"] else END

def worker(name, fails_first=False):
    def node(state):
        attempts = {**state.get("attempts", {}), name: state.get("attempts", {}).get(name, 0) + 1}
        done = not (fails_first and attempts[name] == 1)  # fails_first가 True이면 첫 번째 시도에서 실패하고, Supervisor가 같은 작업을 다시 선택합니다.
        return {"attempts": attempts, "order": [f"{name}#{attempts[name]}:{'완료' if done else '실패'}"], "pending": [p for p in state["pending"] if p != name or not done]}
    return node

graph = StateGraph(SupervisorState)
graph.add_node("supervisor", choose)
graph.add_node("billing", worker("billing"))
graph.add_node("policy", worker("policy", fails_first=True))
graph.add_edge(START, "supervisor")
graph.add_conditional_edges("supervisor", route)
graph.add_edge("billing", "supervisor")
graph.add_edge("policy", "supervisor")
supervisor_skeleton = graph.compile()
show_graph(supervisor_skeleton)
print("실행 순서:", supervisor_skeleton.invoke({"pending": ["billing", "policy"]})["order"])
print("제어 흐름 특성:", flows_of(supervisor_skeleton), "| 모델 호출 0회")

In [ ]:
# Hierarchical(계층형) 구조를 서브그래프로 구성하여 팀 내부 로그와 상위 계층 보고서의 크기 차이를 확인합니다.
def merge_reports(left, right):
    return {**(left or {}), **(right or {})}

class TopState(TypedDict, total=False):
    reports: Annotated[dict, merge_reports]

class TeamState(TopState, total=False):
    log: list[dict]  # 하위 팀 서브그래프 내부에서만 사용하는 필드입니다. TopState에 없는 필드는 상위 계층으로 전달되지 않습니다.

team_desk, TEAM_LOGS = make_desk(double_pay, label="hierarchy"), {}
LIMITS = {"결제팀": ["환불 실행 여부는 알 수 없습니다."], "규정팀": ["검색 상위 조각만 확인했으므로 개별 약정이 누락되었을 수 있습니다."]}

def team(name, collect):
    # 자료 수집(gather) 후 핵심 근거와 한계만 요약해 보고(report)하는 팀 서브그래프를 생성합니다.
    def gather(state):
        return {"log": [collect()]}
    def report(state):
        TEAM_LOGS[name] = state["log"]
        refs = re.findall(r"F-P-\d+|E-[0-9a-f]{12}", json.dumps(state["log"], ensure_ascii=False))
        return {"reports": {name: {"근거": sorted(set(refs)), "상태": "complete" if refs else "uncertain", "한계": LIMITS[name]}}}
    sub = StateGraph(TeamState)
    sub.add_node("gather", gather); sub.add_node("report", report)
    sub.add_edge(START, "gather"); sub.add_edge("gather", "report")
    return sub.compile()

billing_tool, policy_tool = team_desk.tools("billing")[0], team_desk.tools("policy")[0]
top = StateGraph(TopState)
top.add_node("결제팀", team("결제팀", lambda: billing_tool.invoke({"invoice_id": "INV-2026-08"})))
top.add_node("규정팀", team("규정팀", lambda: policy_tool.invoke({"query": "서비스 크레딧 신청 기한"})))
top.add_edge(START, "결제팀"); top.add_edge("결제팀", "규정팀")
hierarchy = top.compile()
show_graph(hierarchy, xray=True)
final_top = hierarchy.invoke({"reports": {}})
reports = final_top["reports"]
size = lambda value: len(json.dumps(value, ensure_ascii=False).encode("utf-8"))
show_table([{"팀": n, "전체 로그(바이트)": size(TEAM_LOGS[n]), "위로 올린 전달물(바이트)": size(r), "전달물": str(r)[:70]} for n, r in reports.items()])
print("최상위 상태의 필드:", sorted(final_top), "| 모델 호출 0회. 도구는 저장소 내부의 데이터와 BM25 인덱스만 조회합니다.")

다음 코드 셀의 비교는 단순히 "계획이 있는가 없는가"의 비교가 아닙니다. 두 실행기 모두 동일한 초기 계획(`PLAN`)을 가지고 시작합니다.

**먼저 예측하세요.** 총 5단계 계획 중 2단계인 `규정 검색`이 실패한다고 가정해 봅시다. 각 단계 실행 후 결과를 즉시 확인하지 않는 **고정 계획** 실행기는 이 실패를 몇 번째 단계에 가서야 알아챌까요? 그리고 그 사이에 낭비한 단계는 몇 개일까요? 매 단계 결과를 관찰한 뒤 계획을 수정하는 **재계획(replan)** 실행기와 비교하여 예상 숫자를 적어 본 뒤 아래 셀을 실행해 보세요.

In [ ]:
# 실패 재현: 2단계에서 검색이 실패할 때 고정 계획 실행기와 재계획 실행기의 낭비 단계 수를 비교합니다.
PLAN = ["결제 조회", "규정 검색", "가용률 조회", "크레딧 계산", "답 작성"]

def do_step(step):
    return step != "규정 검색"  # 기본 '규정 검색'은 실패하고, '규정 검색(넓은 검색어)'로 수정하면 성공합니다.

def fixed(plan):
    # 중간 확인 없이 끝까지 실행한 뒤 마지막 검증 단계에서야 실패를 발견하는 고정 계획 방식입니다.
    log = [{"단계": step, "성공": do_step(step)} for step in plan]
    first_bad = next(i for i, r in enumerate(log) if not r["성공"])
    noticed = len(log)  # 마지막 검증 단계에 도달해서야 실패 사실을 확인합니다.
    return {"실행 방식": "고정 계획", "실행한 단계 수": len(log), "실패를 알아챈 단계": noticed, "낭비한 단계 수": noticed - first_bad - 1, "끝난 상태": "마지막 검증에서 근거 없음 발견"}

def replan(plan):
    # 매 단계 실행 직후 결과를 확인하여 실패 시 즉시 검색 범위를 넓힌 단계로 재계획하는 방식입니다.
    queue, ran, noticed = list(plan), 0, None
    while queue:
        step, queue = queue[0], queue[1:]
        ran += 1
        if not do_step(step):
            noticed = noticed or ran  # 단계 실행 직후 결과를 바로 확인합니다.
            queue.insert(0, "규정 검색(넓은 검색어)")
    return {"실행 방식": "재계획", "실행한 단계 수": ran, "실패를 알아챈 단계": noticed, "낭비한 단계 수": 0, "끝난 상태": "수정한 계획으로 완료"}

show_table([fixed(PLAN), replan(PLAN)])
print("모델 호출 0회")

**관찰할 것.** 먼저 Supervisor 뼈대의 `실행 순서` 출력에서 `supervisor` 노드가 매번 Worker 실행 사이마다 끼어 있는 모습을 확인해 보세요. `policy#1:실패` 직후에 Supervisor로 제어권이 돌아오고, 동일한 `policy` 작업이 다시 선택되어 `policy#2:완료`로 이어집니다. 3절의 실제 Supervisor에서는 이 재시도 선택이 부모 모델의 응답 안에서 일어났고, 지금 본 함수 뼈대에서는 `route` 함수와 `pending` 상태 리스트에서 일어났습니다. 선택 로직을 코드로 옮기면 실행 순서와 재시도 조건을 명확히 읽고 수정할 수 있는 대신, 상태 스키마와 분기 경로를 개발자가 직접 설계해야 하며 미리 정의하지 않은 예외 경로는 처리하지 못합니다. `flows_of` 출력에 표시된 `순환`이 바로 이 루프 구조입니다. 이 뼈대를 통해 Supervisor 패턴의 제어 흐름 정체를 확인할 수 있습니다. 바로 **조건부 분기 + 루프**의 조합이며, 완전히 새로운 제어 구조가 아닙니다. 다음 노트북(03a, 03b)에서 여러 패턴 이름을 이 네 가지 기본 흐름으로 번역해 봅니다.

두 번째 Hierarchical 표에서는 하위 팀의 `전체 로그(바이트)`와 상위 계층으로 보고된 `위로 올린 전달물(바이트)`의 크기를 비교해 보세요. 상위 계층의 Context 부담이 크게 줄어드는 것이 이득이고, 요약 과정에서 빠진 세부 내용을 상위 계층이 알 수 없게 되는 것이 대가입니다. 보고서에 `한계` 항목을 별도로 둔 이유가 바로 여기에 있습니다. 따라서 시스템에 계층을 하나 더 추가할 때는 반드시 먼저 자문해 봐야 합니다. **"이 중간 계층이 독자적으로 판단하고 걸러낼 일이 있는가?"** 만약 판단할 일이 없다면 정보 전달 손실과 응답 지연만 늘어납니다.

세 번째 Plan-and-Execute 표에서는 `실패를 알아챈 단계`와 `낭비한 단계 수`를 비교해 보세요. 두 방식 모두 초기 계획을 세운다는 점은 같습니다. 핵심 차이는 **각 단계의 실행 결과를 관찰한 뒤 계획을 수정(replan)할 수 있는가**입니다. 재계획 방식은 `실행한 단계 수`(6단계)가 고정 계획(5단계)보다 오히려 많습니다. 하지만 단계를 한 번 더 수행한 덕분에 근거 없는 잘못된 답변을 만들지 않고 정상 완료에 도달했습니다. 단순히 실행 단계 수만 보고 효율을 판단해서는 안 되는 이유가 여기에 있습니다.

## 6. 실제 Trace 비교

**문제.** 앞서 정리한 아키텍처별 특징이 실제 모델 실행 결과에서도 그대로 나타날까요? 구조의 이름만 믿지 않고 앞서 실행한 Single, Supervisor, Handoff의 실제 trace 측정값을 한 자리에 모아 비교해 봅니다. 비교 기준은 **누가 다음 행동을 결정했는가(`control owner`)**, **비용이 얼마나 들었는가**, **Context가 어디까지 전달되었는가**, 그리고 **누가 최종 응답을 작성했는가**입니다.

**먼저 예측하세요.** 세 가지 실행(Single, Supervisor, Handoff 턴 1) 중 입력 토큰 합계가 가장 큰 구조는 무엇일까요? 또한 크레딧 계산 기준이 실제 결제 내역의 기본 이용료분과 어긋날 위험이 가장 큰 구조는 무엇이며, 그 이유는 무엇일까요? 예상 답안을 적어 본 뒤 아래 두 셀을 실행하여 확인합니다. 동일한 초기 요청(T-1001)끼리 공정하게 비교하기 위해 Handoff는 턴 1의 결과만 포함합니다. 표의 `전달 글자 수`는 구조마다 전달 경계의 정의가 다르므로 단순 크기 비교용이 아니라 전달 방식을 구분하는 참고 값으로 읽어야 합니다(Single은 마지막 호출의 전체 입력, Supervisor는 `task` 위임 설명과 반환된 텍스트의 합, Handoff는 다음 담당자에게 넘어간 `notes` 메모의 길이입니다).

In [ ]:
# Single, Supervisor, Handoff(턴 1)의 비용 지표(COST)와 Context 경계 지표(BOUNDARY)를 표로 비교합니다.
handoff_owners = [e["actor"] for e in turn1["events"] if e["kind"] in {"handoff", "owner_retained"}]
handoff_speaker = next((e["actor"] for e in turn1["events"] if e["kind"] == "final_answer"), "없음")
delegated = sum(len(c["args"].get("description", "")) for m in supervisor_messages for c in getattr(m, "tool_calls", []) if c["name"] == "task") + sum(map(len, returned.values()))
comparison = [
    row_of("Single", single, single_desk, single_desk.trace.events, ["agent"], "agent", "전체 대화와 모든 도구 결과가 한 Context에 누적", sum(split_last_input(single["rec"]).values())),
    row_of("Supervisor", supervisor_run, supervisor_desk, supervisor_desk.trace.events, ["supervisor"], "supervisor", "Worker에는 task 설명만, 부모에는 마지막 메시지만 전달", delegated),
    row_of("Handoff", turn1, handoff_desk, turn1["events"], handoff_owners, handoff_speaker, "다음 담당자에는 질문, 메모, 최근 대화 2개만 전달", len(turn1["state"].get("notes") or "")),
]
COST = ["구조", "control owner", "모델 호출", "입력 토큰", "캐시 읽기", "usage 누락", "시간(초)", "최종 응답 주체", "실행 오류"]
BOUNDARY = ["구조", "Context 전달 범위", "전달 글자 수", "outcome", "크레딧 계산 기준", "답의 금액", "검증 문제"]
show_table(comparison, COST)
show_table(comparison, BOUNDARY)
print("모델 호출 0회. 각 행은 1회 실행 관찰 결과입니다.")

In [ ]:
def steps_of_messages(messages, agent):
    # 메시지 목록에서 에이전트가 수행한 도구 호출 및 최종 응답 단계를 순서대로 추출합니다.
    steps = []
    for message in messages:
        if isinstance(message, AIMessage):
            calls = [f"task({c['args'].get('subagent_type')})" if c["name"] == "task" else c["name"] for c in message.tool_calls]
            steps.append(f"{agent}: {', '.join(calls)}" if calls else f"{agent}: 최종 응답")
    return steps

def steps_of_events(events):
    # Handoff 이벤트 목록에서 담당자 전환 및 최종 응답 흐름을 순서대로 추출합니다.
    words = {"final_answer": "최종 응답", "handoff_limit": "상한 도달"}
    return [f"{e['actor']}: {words.get(e['kind']) or e.get('owner') + '에게'}" for e in events if e["kind"] in CONTROL]

# 세 구조의 실행 단계 흐름과 제어권이 처음 갈라지는 지점을 비교합니다.
SEQUENCES = {"Single": steps_of_messages(single["state"].get("messages", []), "agent"), "Supervisor": steps_of_messages(supervisor_messages, "supervisor"), "Handoff": steps_of_events(turn1["events"])}
for name, steps in SEQUENCES.items():
    print(f"{name}:", " → ".join(steps))
def first_difference(a, b):
    index = next((i for i, (x, y) in enumerate(zip(a, b, strict=False)) if x != y), min(len(a), len(b)))
    return {"순번": index + 1, "왼쪽": a[index] if index < len(a) else "(끝)", "오른쪽": b[index] if index < len(b) else "(끝)"}
show_table([{"비교": f"{x} / {y}", **first_difference(SEQUENCES[x], SEQUENCES[y])} for x, y in (("Single", "Supervisor"), ("Supervisor", "Handoff"))])
owner_change = next((e for e in turn1["events"] if e["kind"] == "handoff" and e.get("previous_owner")), None)
print("담당자가 바뀐 첫 사건:", (owner_change["previous_owner"], "→", owner_change["owner"]) if owner_change else "이번 턴에는 없음", "| 모델 호출 0회")

**관찰할 것.** 첫 번째 표에서 `control owner`와 `최종 응답 주체` 열을 나란히 읽어 보세요. Single과 Supervisor 구조에서는 다음 행동을 결정한 주체와 최종 응답을 작성한 주체가 동일합니다. 반면 Handoff 구조에서는 어떻게 나타났나요? 바로 이 두 열의 차이가 "조사 결과를 돌려받는 구조(Supervisor)"와 "대화 제어권을 넘기는 구조(Handoff)"의 차이를 실제 실행 데이터로 보여 주는 지점입니다. 이어서 두 번째 셀의 실행 단계 순서 표에서 세 구조의 제어권이 처음으로 갈라지는 순번을 확인해 보세요. Supervisor가 `task` 도구를 호출하는 순간 실제 조사 작업은 Worker로 넘어가지만, 조사가 끝난 뒤 다음 행동을 결정하는 제어권은 다시 부모(`supervisor`)에게 돌아옵니다. 반면 Handoff에서 담당자(owner)가 한 번 넘어가면 이전 담당자에게 제어권이 되돌아오지 않습니다.

두 번째 표에서는 `크레딧 계산 기준`과 `답의 금액`을 구조별로 비교해 보세요. 만약 구조마다 계산된 금액이나 기준이 다르다면, 그 원인을 바로 왼쪽의 `Context 전달 범위` 차이로 설명할 수 있는지 확인해 보세요. Single과 Supervisor에서는 크레딧을 계산하는 주체가 결제 내역 원문을 직접 조회하거나 Worker가 반환한 보고서에서 읽습니다. 반면 Handoff의 `technical` 담당자는 앞선 `billing` 담당자가 남겨 준 인계 메모(`notes`)에 의존합니다. 만약 세 구조의 금액이 모두 일치했다면, 이번 1회 실행에서는 인계 메모에 필요한 숫자가 빠지지 않아 전달 손실이 겉으로 드러나지 않은 것입니다.

마지막으로 이 비교표를 통해 **주장할 수 있는 것**과 **주장할 수 없는 것**을 명확히 구분해야 합니다. 이 표로 말할 수 있는 것은 "이번 1회 실행에서 각 구조가 어떤 측정값을 기록했는가"와 "어떤 구조적 경로를 통해 특정 정보 손실이나 비용 증가가 발생할 수 있는가"입니다. 반면 이 표만 보고 "특정 구조가 항상 더 우수하다"라고 결론 내릴 수는 없습니다. 단일 사례(T-1001)에 대한 1회 실행 결과일 뿐이며, 모델의 비결정성 때문에 실행할 때마다 세부 수치가 달라질 수 있기 때문입니다. 여러 사례와 반복 실행을 통해 아키텍처를 선정하는 정량 측정은 05 노트북에서 다룹니다. 또한 두 구조를 배타적으로 하나만 골라야 하는 것도 아닙니다. Handoff로 대화를 넘겨받은 담당자 내부에서 다시 병렬 조사 Workflow를 실행하는 식으로 조합할 수 있습니다. 누구와 대화할지 정하는 문제(담당자 전환)와 내부 조사를 어떻게 수행할지 정하는 문제(병렬 위임)는 서로 다른 설계 축이기 때문입니다.

## 직접 해보기

아래 네 가지 실습 과제는 모두 **한 번에 조건을 하나씩만** 바꾸어 결과를 비교합니다. 유료 모델을 호출하는 해보기 1과 해보기 2는 기본적으로 실행이 꺼져 있습니다(`RUN_EXERCISE = False`). 코드를 수정한 뒤 `RUN_EXERCISE`를 `True`로 바꾸어 실행해 보세요. 모델을 호출하지 않는 해보기 3과 해보기 4는 그대로 실행할 수 있습니다. 실행 결과를 정리할 때는 막연히 "성능이 좋아졌다/나빠졌다"라고 적는 대신, **표의 어느 항목이 어떻게 달라졌는지** 구체적인 수치와 상태로 적어 보세요.

### 해보기 1

(실제 모델 호출) Worker의 역할 설명(`description`)을 의도적으로 모호하게 바꾸어 봅니다. Supervisor의 부모 모델은 이 설명을 읽고 누구에게 어떤 작업을 맡길지 결정합니다. 설명을 모호하게 바꿨을 때 표의 `위임 순서`와 `메시지당 task 수`가 어떻게 달라지는지 확인해 보세요. 또한 표 아래 출력되는 `실제 모델 호출`과 `예산이 센 호출`의 차이가 바로 예산 통제 밖에서 발생한 모델 호출입니다. 만약 부모가 혼란을 겪어 기본 `general-purpose` Worker 등이 개입했다면 이 차이가 어떻게 나타나는지도 살펴보세요. Worker 내부에서 호출한 세부 도구를 확인하고 싶다면 `vague["rec"].tool_results`를 3절의 표처럼 출력하는 코드를 직접 추가해 보세요.

In [ ]:
RUN_EXERCISE = False
VAGUE_DESCRIPTION = "정보를 찾는다."  # 이 설명 문자열을 다양하게 수정해 보세요.

if not RUN_EXERCISE:
    print("실행하지 않았습니다. 값을 수정한 뒤 RUN_EXERCISE를 True로 바꾸어 실행해 보세요.")
else:
    announce("해보기 1: Worker 설명을 모호하게 변경하여 실행", model_calls=14)
    vague_desk = make_desk(double_pay, label="vague")
    vague_specs = [{**subagent_spec(role, vague_desk, model), "description": VAGUE_DESCRIPTION} for role in ("billing", "policy", "outage")]
    vague_app = create_deep_agent(
        model=model, tools=vague_desk.tools("writer"), subagents=vague_specs, name="supervisor", permissions=vague_specs[0]["permissions"],
        system_prompt=f"고객 요청을 조사 Worker에게 task로 위임하고 결과로 최종 응답합니다. {scope_line(vague_desk)}\n금액 계산은 calculate_credit을 쓰세요.\n{OUTCOME_GUIDE}",
        middleware=allowlist_middleware(vague_desk.budget, frozenset({"task"}) | ROLE_TOOLS["writer"]), response_format=ProviderStrategy(FinalAnswer))
    vague = execute(vague_app, ask(double_pay["question"]), Recorder(vague_desk))
    show_table([delegation_row("설명 원본", supervisor_run), delegation_row("설명 모호", vague)])
    counted = vague_desk.budget.report()["counts"].get("model", 0)
    print("실제 모델 호출:", vague["rec"].totals()["model_calls"], "| 예산이 센 호출:", counted, "| 오류:", vague["error"] or "없음")
    print("크레딧 계산 기준:", credit_basis(vague_desk, vague_desk.trace.events), "| 답의 금액:", amounts_in(vague["final"]))

### 해보기 2

(실제 모델 호출) Worker에게 조회할 사실 데이터를 `WorkerTask.facts` 필드로 미리 넣어 주면 모델 호출 횟수와 도구 호출 횟수가 줄어드는지 확인합니다. `facts`에 넣어 준 값은 Worker가 별도 도구를 호출하지 않아도 프롬프트 안에서 바로 읽을 수 있습니다. 실행 후 표의 `모델 호출`, `입력 토큰`, `도구 호출`, `근거 수`, `검증 문제` 열을 비교해 보세요. 만약 도구 호출이 줄어들었다면 그 대가로 무엇을 잃었나요? 반대로 사실을 미리 주었는데도 호출 횟수가 줄지 않았다면, 모델이 전달받은 사실을 그대로 쓰지 않고 조회 도구를 다시 호출한 것입니다. 왜 모델이 도구를 다시 호출했는지 `TASK_CONTRACT`의 근거(Evidence) 인용 규칙과 연결하여 이유를 생각해 보세요. 도구를 통해 정식 근거로 등록되지 않은 사실을 최종 결과에 인용할 수 있을까요?

In [ ]:
RUN_EXERCISE = False
GOALS = {"billing": "청구서 INV-2026-08의 중복 결제 여부를 확인하세요.", "outage": "2026-08 가용률과 장애 사건을 확인하세요."}  # 작업 목표 문구를 수정해 보세요.

if not RUN_EXERCISE:
    print("실행하지 않았습니다. 값을 수정한 뒤 RUN_EXERCISE를 True로 바꾸어 실행해 보세요.")
else:
    announce("해보기 2: Worker에 facts 사전 전달 유무 비교", model_calls=12)
    rows = []
    for role, goal in GOALS.items():
        for given in (False, True):
            desk = make_desk(double_pay, label=f"facts-{role}")
            facts = {"billing": {"payments": desk.payments()}, "outage": load_outage("알파")}[role] if given else {}
            recorder = Recorder(desk)
            result = run_worker(WorkerTask(task_id=role, goal=goal, facts=facts), desk, model, config={"callbacks": [recorder]})
            totals = recorder.totals()
            rows.append({"역할": role, "facts 전달": given, "모델 호출": totals["model_calls"], "입력 토큰": totals["input_tokens"], "도구 호출": desk.budget.report()["counts"].get("tool", 0),
                         "상태": result.status, "근거 수": len(result.evidence_refs), "검증 문제": validate_result(result, desk.known_refs())})
    show_table(rows)

### 해보기 3

Handoff의 왕복 상한(`LIMITS_TO_TRY`)을 `0`, `1`, `3`으로 바꾸어 4절의 왕복 상한 실험 표를 다시 출력해 봅니다. 표의 `넘긴 횟수`와 `경로`를 확인하고, 상한이 `0`인 설정(한 번도 다른 담당자에게 넘기지 못하고 즉시 확인 요청으로 종료)은 어떤 성격의 업무에서 적합한 값일지 생각해 보세요. 만약 실제 고객 응대 서비스를 운영한다면 이 왕복 상한을 몇 회로 설정할지, 그리고 왜 그 숫자를 선택했는지 한 문장으로 정리해 보세요.

In [ ]:
# 왕복 상한 값을 바꾸어 가며 종료 상태와 이동 경로가 어떻게 달라지는지 확인합니다.
LIMITS_TO_TRY = [0, 1, 3]  # 리스트의 상한 값들을 자유롭게 수정해 보세요.
rows = []
for limit in LIMITS_TO_TRY:
    result = build_bounce(limit=limit).invoke({}, config={"recursion_limit": 25})
    rows.append({"왕복 상한": limit, "종료 상태": result["status"], "넘긴 횟수": result.get("bounces", 0), "경로": " → ".join(result["path"])})
show_table(rows)
print("모델 호출 0회")

### 해보기 4

5절에서 살펴본 Hierarchical(계층형) 구조의 팀 보고서(`reports`)에서 특정 필드를 제거해 봅니다. 기본 설정은 `한계` 필드를 제거하도록 되어 있습니다(`REMOVE_FIELD = "한계"`). 보고서에서 특정 필드를 지우면 상위 계층이 답할 수 없게 되는 질문이 두 번째 표에 `False`로 표시됩니다. 전달물의 바이트 크기가 줄어든 대가로 상위 계층이 어떤 질문에 답할 능력을 잃었는지 확인해 보세요. 또한 `"근거"`나 `"상태"`로 값을 바꾸어 실행해 보고, 본인의 실제 업무에서는 어느 항목이 절대 빠져서는 안 되는 필수 항목인지 판단해 보세요.

In [ ]:
# 하위 팀이 올리는 보고서에서 특정 필드를 제거했을 때 줄어드는 크기와 답할 수 없게 되는 질문을 확인합니다.
REMOVE_FIELD = "한계"  # "근거" 또는 "상태"로 값을 바꾸어 실행해 보세요.
QUESTIONS = {"환불 대상의 근거를 원문까지 따라갈 수 있습니까?": "근거", "이 팀의 결과를 믿고 바로 답해도 됩니까?": "상태", "이 팀이 보지 못한 것은 무엇입니까?": "한계"}

trimmed = {name: {k: v for k, v in report.items() if k != REMOVE_FIELD} for name, report in reports.items()}
show_table([{"팀": name, "원래 바이트": size(reports[name]), "지운 뒤 바이트": size(trimmed[name])} for name in reports])
show_table([{"위층의 질문": question, "필요한 칸": field, "답할 수 있음": all(field in r for r in trimmed.values())} for question, field in QUESTIONS.items()])
print("모델 호출 0회")

In [ ]:
# 본인 업무에 적합한 아키텍처를 하나 고르고, 단순 코드나 Skill 대신 해당 구조가 필요한 이유를 설계 기록으로 저장합니다.
MY_ROW = "Supervisor"  # 1절의 ARCHITECTURES 표에서 본인 업무에 맞는 구조 이름으로 바꾸어 보세요.
WHY_NOT_CODE_OR_SKILL = ("(예시입니다. 본인의 설명으로 수정해 보세요) 크레딧 곱셈과 환불 게이트는 일반 코드로 충분하고 절차 안내는 Skill로 충분합니다. "
                         "SubAgent가 필요한 이유는 서로 독립적인 조사 결과를 하나의 Context에 섞여 쌓이지 않게 격리하기 위해서입니다.")

chosen = next(row for row in ARCHITECTURES if row["구조"] == MY_ROW)
show_table([chosen])
path = write_design_log("Agent·SubAgent의 필요성: 코드나 Skill로 충분하지 않은 이유는?", f"{MY_ROW}: {WHY_NOT_CODE_OR_SKILL}")
print("설계 기록을 저장했습니다:", path.name, "| 모델 호출 0회")

## 정리

1절에서 보았던 아키텍처 변화의 지도를 이번 실습에서 직접 측정한 지표들로 다시 정리해 봅니다. 각 구조마다 **그 구조를 도입하게 만든 직전 구조의 한계**와 **새로 치른 비용**이 출력 표의 어느 항목으로 확인되었는지 함께 대조해 보세요.

| 구조 | 한계를 확인한 측정 항목 | 새로 치른 비용을 확인한 측정 항목 |
|---|---|---|
| Single | 호출별 `입력 토큰`의 누적 증가, 마지막 호출 입력의 구성 요소 분해 | 모델 호출 횟수와 순서를 실행 전에 미리 고정할 수 없음 |
| Supervisor | `안에서 본 도구 결과 글자`와 `부모에게 돌아온 글자`의 크기 차이 (Context 분리) | 총 `모델 호출` 수 증가, `메시지당 task 수`가 지침 한 줄에 흔들림, Worker마다 예산(budget)·도구 제한을 별도로 걸어야 함 |
| Handoff | 앞선 담당자가 끝까지 중계하지 않아도 됨, `최종 응답 주체`의 이동 | 인계 메모 누락에 따른 `크레딧 계산 기준` 오류 위험, 왕복 상한(`MAX_HANDOFFS`) 및 담당자 유지 판단 비용 |
| Supervisor 뼈대, Hierarchical, Plan-and-Execute | 일반 함수 뼈대의 명확한 실행 순서와 상태 분리 | 하위 팀 `전체 로그`와 상위 `전달물` 사이의 정보 손실, 고정 계획의 `낭비한 단계 수` |

이번 노트북에서 확인한 세 가지 핵심 원칙을 기억해 두세요.
1. **측정 기반 전환**: 아키텍처를 더 복잡한 구조로 바꾸는 이유는 유행이 아니라 실제로 측정된 직전 구조의 한계여야 합니다.
2. **경계와 전달 계약**: 에이전트를 나눌 때마다 새로운 경계가 생기며, 경계를 넘어가는 핵심 값(예: 기본 이용료분, 근거 참조 번호)은 코드나 명시적 계약이 보장하지 않으면 언제든 요약 과정에서 사라질 수 있습니다.
3. **코드로 고정할 영역 구분**: 프롬프트 한 줄로 바뀌는 실행 구조는 고정된 아키텍처가 아닙니다. 업무에서 반드시 고정해야 하는 실행 흐름이 무엇인지는 다음 노트북(03a, 03b)에서 네 가지 제어 흐름 단위로 명확히 설계합니다.

## 확인 질문

1. Supervisor 구조에서 하위 Worker의 내부 도구 호출 대화가 부모 에이전트에게 보이지 않는 설계는 어떤 이득과 어떤 대가를 가져오나요? 이번 실습 출력 표의 어느 항목이 그 근거인가요?
2. 3절의 실패 재현에서 시스템 지침의 "동시에"라는 문구를 뺐을 때 병렬 위임이 사라졌다면, 그것은 코드 버그일까요, 아니면 누구의 결정일까요?
3. Supervisor와 Handoff 구조에서 고객에게 전달되는 최종 응답은 각각 누가 작성하나요? 6절 비교표의 어느 열에서 이를 확인할 수 있나요?
4. Handoff의 후속 턴(턴 2)에서 담당자(owner)를 그대로 유지(`owner_retained`)했는데도 왜 현재 질문에 대한 규정 근거를 다시 검색해야 할까요? 또한 턴 1의 `크레딧 계산 기준` 항목은 담당자 간 전달 손실 문제와 어떻게 연결되나요?
5. Hierarchical 구조처럼 시스템에 중간 계층을 하나 더 추가할 때 반드시 먼저 확인해야 할 질문은 무엇이며, 5절의 어느 측정 항목이 그 근거인가요?
6. 프레임워크가 자동으로 끼워 넣은 `general-purpose` Worker가 이번 T-1001 사례에서 실제로 수행할 수 있는 조사 업무는 무엇인가요? 3절 비교표의 어느 항목을 근거로 그렇게 판단할 수 있나요?

다음 노트북 **03a**와 **03b**에서는 지금까지 살펴본 아키텍처 패턴들을 패턴 이름 없이 바닥부터 다시 조립합니다. 순차, 병렬, 분기, 루프라는 네 가지 기본 흐름을 `StateGraph`로 직접 구현하고 실패를 재현한 뒤, 오늘 다룬 아키텍처들이 각각 어떤 기본 흐름의 조합인지 명확하게 번역해 봅니다.